# ASQA — Dataset Profile

This notebook describes the **ASQA dev dataset** used in the Context Matters
project.

The goal is to understand the dataset itself:

- what makes an ASQA question ambiguous;
- how one ambiguous question maps to multiple disambiguated QA pairs;
- how short-answer aliases are represented;
- how long-form answer annotations are structured;
- how Wikipedia grounding is represented;
- how complete that grounding is;
- why ASQA is useful for evaluating diversity-aware retrieval.

This notebook does **not** report Sprint-1, Sprint-2, or Sprint-3 model results.

## 1. Frozen Dataset Source

The project uses:

- Dataset: `din0s/asqa`
- Split: `dev`
- Frozen revision: `084060f16b46f3165318f760b2339208b19a0bde`
- Questions: **948**

The full dev split is treated as `PROJECT_PROTECTED_FINAL` in this project.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO = Path.cwd().resolve()

while REPO != REPO.parent and not (REPO / "docs").is_dir():
    REPO = REPO.parent

assert (REPO / "docs").is_dir(), "Could not locate repository root"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)

In [ ]:
from datasets import load_dataset, disable_progress_bar

disable_progress_bar()

SOURCE = "din0s/asqa"
REVISION = "084060f16b46f3165318f760b2339208b19a0bde"

dataset = load_dataset(
    SOURCE,
    split="dev",
    revision=REVISION,
    download_mode="reuse_dataset_if_exists",
)

rows = tuple(dataset)

assert len(rows) == 948
assert len({row["sample_id"] for row in rows}) == 948

print("ASQA validation: PASS")
print(f"Questions:       {len(rows):,}")
print(f"Source:          {SOURCE}")
print("Split:           dev")
print(f"Frozen revision: {REVISION}")
print("Unique IDs:      948")
print("Evidence role:   PROJECT_PROTECTED_FINAL")
print("LLM/API calls made: 0")

## 2. Dataset Schema

Each ASQA example contains five top-level fields.

| Field | Meaning |
|---|---|
| `ambiguous_question` | Original ambiguous question |
| `qa_pairs` | Disambiguated QA aspects |
| `wikipages` | Wikipedia pages associated with the question |
| `annotations` | Long-form answer annotations and supporting knowledge |
| `sample_id` | Stable sample identifier |

The `qa_pairs` field is central to ASQA because one ambiguous question can
correspond to several legitimate interpretations.

In [ ]:
schema = pd.DataFrame(
    [
        ["ambiguous_question", "string", "Original ambiguous question"],
        ["qa_pairs", "list[object]", "Disambiguated QA aspects"],
        [
            "qa_pairs.question",
            "string",
            "Disambiguated question for one aspect",
        ],
        [
            "qa_pairs.short_answers",
            "list[string]",
            "Official short-answer aliases for one aspect",
        ],
        [
            "qa_pairs.context",
            "string",
            "Optional annotated context for one aspect",
        ],
        [
            "qa_pairs.wikipage",
            "string / null",
            "Optional Wikipedia page grounding",
        ],
        ["wikipages", "list[object]", "Question-associated Wikipedia pages"],
        [
            "annotations",
            "list[object]",
            "Long-form human annotations",
        ],
        [
            "annotations.long_answer",
            "string",
            "Human-written long-form answer",
        ],
        [
            "annotations.knowledge",
            "list[object]",
            "Supporting knowledge used by an annotation",
        ],
        ["sample_id", "string", "Stable sample identifier"],
    ],
    columns=["Field", "Type", "Description"],
)

print(schema.to_string(index=False))

## 3. What Makes ASQA Different?

ASQA focuses on **ambiguous questions**.

A short question can have several valid interpretations.

For example:

`Who has the highest goals in world football?`

can refer to:

- men's international football;
- all-time men's football;
- women's international football.

Instead of forcing one interpretation, ASQA represents these separately as
multiple disambiguated QA pairs.

A high-quality long-form answer should resolve the ambiguity and cover the
relevant distinct interpretations.

## 4. Example Ambiguous Question

In [ ]:
example = rows[0]

print("Sample ID:")
print(example["sample_id"])

print("\nAmbiguous question:")
print(example["ambiguous_question"])

print("\nNumber of QA-pair aspects:")
print(len(example["qa_pairs"]))

for i, pair in enumerate(example["qa_pairs"], start=1):
    print("\n" + "=" * 70)
    print(f"ASPECT {i}")
    print("Disambiguated question:")
    print(pair["question"])

    print("Short-answer aliases:")
    print(pair["short_answers"])

    print("Wikipedia page:")
    print(pair["wikipage"])

    print("Annotated context available:")
    print(pair["context"] != "No context provided")

## 5. Number of Aspects per Question

Each official `qa_pair` represents one distinct ASQA aspect.

Questions can contain different numbers of legitimate interpretations.

In [ ]:
aspect_counts = pd.Series(
    [len(row["qa_pairs"]) for row in rows],
    name="QA-pair aspects",
)

print(
    aspect_counts
    .describe()
    .round(2)
    .to_string()
)

print("\nDistribution:")
print(
    aspect_counts
    .value_counts()
    .sort_index()
    .rename_axis("Aspects")
    .to_string()
)

## 6. Total Number of Disambiguated QA Aspects

In [ ]:
total_aspects = int(aspect_counts.sum())

print("Ambiguous questions:", f"{len(rows):,}")
print("Total QA-pair aspects:", f"{total_aspects:,}")
print(
    "Mean aspects per question:",
    round(total_aspects / len(rows), 2),
)

## 7. Short-Answer Aliases

Each aspect can contain multiple official short-answer aliases.

For example, an entity may appear as both a surname and a full name.

These aliases are important because answer evaluation should allow legitimate
surface-form variation without inventing new synonyms.

In [ ]:
alias_counts = []

for row in rows:
    for pair in row["qa_pairs"]:
        alias_counts.append(
            len(
                [
                    alias
                    for alias in pair["short_answers"]
                    if str(alias).strip()
                ]
            )
        )

alias_counts = pd.Series(
    alias_counts,
    name="Aliases per aspect",
)

print(
    alias_counts
    .describe()
    .round(2)
    .to_string()
)

print(
    "\nTotal non-empty official alias entries:",
    f"{alias_counts.sum():,}",
)

## 8. Aspect-Level Wikipedia Grounding

Some QA-pair aspects provide an annotated Wikipedia page and context.

Others explicitly contain:

`No context provided`

This means aspect-level document grounding is incomplete in the public ASQA
data.

In [ ]:
aspect_grounding = []

for row in rows:
    for pair in row["qa_pairs"]:
        has_context = (
            isinstance(pair["context"], str)
            and pair["context"].strip()
            and pair["context"] != "No context provided"
        )

        has_wikipage = bool(pair["wikipage"])

        aspect_grounding.append(
            {
                "has_context": bool(has_context),
                "has_wikipage": bool(has_wikipage),
            }
        )

grounding_df = pd.DataFrame(aspect_grounding)

grounding_summary = pd.DataFrame(
    [
        [
            "Total aspects",
            len(grounding_df),
            100.0,
        ],
        [
            "Aspects with annotated context",
            int(grounding_df["has_context"].sum()),
            round(100 * grounding_df["has_context"].mean(), 2),
        ],
        [
            "Aspects with annotated Wikipedia page",
            int(grounding_df["has_wikipage"].sum()),
            round(100 * grounding_df["has_wikipage"].mean(), 2),
        ],
    ],
    columns=["Grounding property", "Count", "Percent"],
)

print(grounding_summary.to_string(index=False))

## 9. Question-Level Grounding Completeness

A question is fully aspect-grounded only when **every QA pair** has an
annotated Wikipedia page.

This is different from merely having at least one grounded aspect.

In [ ]:
question_grounding = []

for row in rows:
    pairs = row["qa_pairs"]

    grounded_pages = [
        pair["wikipage"]
        for pair in pairs
        if pair["wikipage"]
    ]

    all_grounded = (
        len(grounded_pages) == len(pairs)
    )

    all_distinct = (
        all_grounded
        and len(set(grounded_pages)) == len(grounded_pages)
    )

    question_grounding.append(
        {
            "sample_id": row["sample_id"],
            "aspect_count": len(pairs),
            "grounded_aspect_count": len(grounded_pages),
            "all_aspects_grounded": all_grounded,
            "all_grounded_pages_distinct": all_distinct,
        }
    )

question_grounding_df = pd.DataFrame(question_grounding)

fully_grounded = int(
    question_grounding_df["all_aspects_grounded"].sum()
)

fully_distinct = int(
    question_grounding_df[
        "all_grounded_pages_distinct"
    ].sum()
)

print(
    "Questions with every aspect Wikipedia-grounded:",
    fully_grounded,
    f"({100 * fully_grounded / len(rows):.2f}%)",
)

print(
    "Questions with a distinct grounded page for every aspect:",
    fully_distinct,
    f"({100 * fully_distinct / len(rows):.2f}%)",
)

print(
    "Fully grounded questions where at least two aspects share a page:",
    fully_grounded - fully_distinct,
)

## 10. Why Grounding Completeness Matters

The incomplete aspect-to-page relation is scientifically important.

It means that assuming:

`one aspect = one unique relevant Wikipedia page`

would not faithfully represent the public ASQA annotations.

Some aspects:

- have no annotated page;
- share a page with another aspect;
- contain only partial grounding information.

This is why retrieval evaluation must use an explicitly defined and frozen
aspect-to-passage matching procedure rather than assuming one unique page per
aspect.

## 11. Wikipedia Pages Associated with Each Question

The top-level `wikipages` field contains Wikipedia pages associated with the
ambiguous question.

This field is separate from the aspect-specific `qa_pairs.wikipage` field.

In [ ]:
wikipage_counts = pd.Series(
    [len(row["wikipages"]) for row in rows],
    name="Question-associated Wikipedia pages",
)

print(
    wikipage_counts
    .describe()
    .round(2)
    .to_string()
)

print(
    "\nTotal question-associated Wikipedia-page entries:",
    f"{wikipage_counts.sum():,}",
)

## 12. Long-Form Answer Annotations

Every frozen ASQA dev question contains two human long-form answer annotations.

These answers attempt to resolve the original ambiguity by covering multiple
relevant interpretations in one coherent response.

In [ ]:
annotation_counts = pd.Series(
    [len(row["annotations"]) for row in rows],
    name="Annotations",
)

print(
    annotation_counts
    .describe()
    .round(2)
    .to_string()
)

print(
    "\nAll questions have exactly two annotations:",
    bool((annotation_counts == 2).all()),
)

## 13. Example Human Long-Form Answers

In [ ]:
print("Ambiguous question:")
print(example["ambiguous_question"])

for i, annotation in enumerate(
    example["annotations"],
    start=1,
):
    print("\n" + "=" * 70)
    print(f"ANNOTATION {i}")
    print(annotation["long_answer"])

    print("\nKnowledge items:")
    print(len(annotation["knowledge"]))

## 14. Annotation Knowledge

Each long-form annotation may additionally contain knowledge snippets linked
to Wikipedia pages.

The number of such snippets varies between annotations.

In [ ]:
knowledge_counts = []

for row in rows:
    for annotation in row["annotations"]:
        knowledge_counts.append(
            len(annotation["knowledge"])
        )

knowledge_counts = pd.Series(
    knowledge_counts,
    name="Knowledge snippets per annotation",
)

print(
    knowledge_counts
    .describe()
    .round(2)
    .to_string()
)

print(
    "\nAnnotations with zero knowledge snippets:",
    int((knowledge_counts == 0).sum()),
)

## 15. Question and Long-Answer Lengths

In [ ]:
question_words = pd.Series(
    [
        len(row["ambiguous_question"].split())
        for row in rows
    ],
    name="Ambiguous-question words",
)

long_answer_words = pd.Series(
    [
        len(annotation["long_answer"].split())
        for row in rows
        for annotation in row["annotations"]
    ],
    name="Long-answer words",
)

length_summary = pd.DataFrame(
    {
        "Question": question_words.describe(),
        "Long answer": long_answer_words.describe(),
    }
).round(2)

print(length_summary.to_string())

## 16. Example of Ambiguity Expansion

The following view shows how the original ambiguous question expands into
multiple explicit interpretations.

In [ ]:
for i, row in enumerate(rows[:3], start=1):
    print("=" * 80)
    print(f"EXAMPLE {i}")
    print("Ambiguous question:")
    print(row["ambiguous_question"])

    print("\nDisambiguated questions:")
    for j, pair in enumerate(row["qa_pairs"], start=1):
        print(f"{j}. {pair['question']}")

    print()

## 17. Dataset Annotations vs Project Retrieval Corpus

Two different sources must not be confused.

### ASQA annotations

The ASQA dataset provides:

- ambiguous questions;
- disambiguated QA pairs;
- short-answer aliases;
- optional Wikipedia grounding;
- long-form answer annotations.

### Context Matters retrieval corpus

Our retrieval systems search the complete frozen DPR Wikipedia collection:

**21,015,324 passages**

Snapshot lineage:

**2018-12-20**

The canonical retrieval/generation passage surface is the exact DPR passage
**body**.

Therefore ASQA's annotation fields describe the benchmark, while BM25, DPR,
Contriever, and ColBERT retrieve from the separate full DPR Wikipedia corpus.

## 18. Why ASQA Is Useful for Diversity-Aware Retrieval

ASQA is especially relevant to the Context Matters research question.

An ambiguous question can require evidence for several legitimate aspects.

A relevance-only retriever may repeatedly return passages about the most
dominant interpretation.

A diversity-aware method may instead expose evidence for several distinct
interpretations.

That creates a direct scientific question:

> Does more diverse retrieved context help the language model resolve more of
> the ambiguity, or does it introduce distracting and unsupported information?

ASQA therefore complements:

- PubMedQA's controlled biomedical decision setting;
- HotpotQA's multi-hop reasoning setting.

## 19. Important Dataset Limitations

Important limitations include:

- aspect-level Wikipedia grounding is incomplete;
- multiple aspects can share the same annotated page;
- annotated contexts are not a complete relevance judgment over the full
  Wikipedia retrieval corpus;
- long-form answer annotations are not exhaustive descriptions of every valid
  answer;
- alias occurrence alone does not guarantee semantic correctness;
- the 948-question dev split is protected-final in this project and must not
  be used to tune scoring rules or retrieval methodology.

## 20. Dataset Summary

In [ ]:
summary = pd.DataFrame(
    [
        ["Dataset", "ASQA"],
        ["Split", "dev"],
        ["Questions", f"{len(rows):,}"],
        ["Total QA-pair aspects", f"{total_aspects:,}"],
        [
            "Mean aspects per question",
            f"{aspect_counts.mean():.2f}",
        ],
        [
            "Aspect range",
            f"{aspect_counts.min()}–{aspect_counts.max()}",
        ],
        ["Human annotations per question", "2"],
        [
            "Every aspect page-grounded",
            f"{fully_grounded:,} / {len(rows):,}",
        ],
        [
            "Distinct page for every aspect",
            f"{fully_distinct:,} / {len(rows):,}",
        ],
        [
            "Project retrieval corpus",
            "21,015,324 DPR Wikipedia passages",
        ],
        [
            "Role in project",
            "Ambiguity / multi-aspect QA dataset",
        ],
    ],
    columns=["Property", "Value"],
)

print(summary.to_string(index=False))

## 21. Reproducibility Check

In [ ]:
checks = pd.DataFrame(
    [
        [
            "Dataset contains exactly 948 questions",
            len(rows) == 948,
        ],
        [
            "All sample IDs are unique",
            len({row["sample_id"] for row in rows}) == 948,
        ],
        [
            "Every question has at least two QA-pair aspects",
            all(len(row["qa_pairs"]) >= 2 for row in rows),
        ],
        [
            "Every question has exactly two annotations",
            all(len(row["annotations"]) == 2 for row in rows),
        ],
        [
            "All ambiguous questions are non-empty",
            all(
                bool(row["ambiguous_question"].strip())
                for row in rows
            ),
        ],
        [
            "All QA-pair questions are non-empty",
            all(
                bool(pair["question"].strip())
                for row in rows
                for pair in row["qa_pairs"]
            ),
        ],
        [
            "Every QA-pair has at least one non-empty alias",
            all(
                any(
                    bool(str(alias).strip())
                    for alias in pair["short_answers"]
                )
                for row in rows
                for pair in row["qa_pairs"]
            ),
        ],
        [
            "All human long answers are non-empty",
            all(
                bool(annotation["long_answer"].strip())
                for row in rows
                for annotation in row["annotations"]
            ),
        ],
    ],
    columns=["Check", "PASS"],
)

assert checks["PASS"].all(), checks[~checks["PASS"]]

print(checks.to_string(index=False))

print("\nPASS: ASQA dataset profile is reproducible")
print("LLM/API calls made by this notebook: 0")

## Reproducibility

This notebook uses the same frozen ASQA dev source and revision as the main
project.

It performs descriptive dataset analysis only.

No LLM/API calls are made, and no protected-final outcome is used to tune
project methodology.